## PMXT API -> Kafka -> Consumer Parse -> ClickHouse 测试

本 notebook 用于验证完整工作流：

- 从 PMXT API 获取 `fetch_markets` 原始数据
- 以 JSON 消息写入 Kafka（topic: `polymarket.markets.raw`）
- 从 Kafka 消费并检查消息结构
- 执行消费解析并写入 ClickHouse（保留 `captured_at`）
- 查询新 schema：`polymarket_market_dim` / `polymarket_market_snapshot` / `polymarket_outcome_snapshot`


In [1]:
import sys
from pathlib import Path

from IPython.display import display
import pandas as pd
import clickhouse_connect

# 让 notebook 里可以 import repo 内的 app 包
repo_root = Path.cwd()
repo_root = repo_root.parent
if (repo_root / "app").exists():
    sys.path.insert(0, str(repo_root))

from app.config import settings as app_settings
from app.clients.kafka_client import KafkaConsumerClient
from app.services.consume_service import consume_service

print("Kafka bootstrap servers:", app_settings.KAFKA_BOOTSTRAP_SERVERS)
print("Kafka raw markets topic:", app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW)

ch_client = clickhouse_connect.get_client(
    host=app_settings.CLICKHOUSE_HOST,
    port=int(app_settings.CLICKHOUSE_PORT),
    username=app_settings.CLICKHOUSE_USERNAME,
    password=app_settings.CLICKHOUSE_PASSWORD,
    database=app_settings.CLICKHOUSE_DATABASE,
)


Kafka bootstrap servers: localhost:9092
Kafka raw markets topic: polymarket.markets.raw


In [7]:
from datetime import datetime, UTC

DO_INGEST = True  # 是否手动触发一次 API -> Kafka
DO_CONSUME_CHECK = True  # 是否检查 Kafka 中的消息
DO_CONSUME_TO_CLICKHOUSE = True  # 是否执行 Consumer Parse -> ClickHouse

if DO_INGEST:
    from app.services.ingestion_service import ingestion_service

    published = ingestion_service.publish_polymarket_markets_to_kafka(
        # query="Trump",
        limit=200,
        sort="volume",
    )
    print("Published market messages to Kafka:", published)
else:
    print("Skip ingestion (DO_INGEST=False).")


if DO_CONSUME_CHECK:
    topic = app_settings.KAFKA_TOPIC_POLYMARKET_MARKETS_RAW
    group_id = f"notebook.kafka.check.latest.{datetime.now(UTC).strftime('%H%M%S')}"

    with KafkaConsumerClient(
        topic=topic,
        group_id=group_id,
        auto_offset_reset="latest",
        enable_auto_commit=False,
        consumer_timeout_ms=3000,
    ) as consumer:
        result = consumer.consume(max_messages_per_partition=100, max_workers=3)

    rows = []
    for partition, messages in result.items():
        for msg in messages:
            rows.append(
                {
                    "partition": partition,
                    "market_id": msg.get("market_id"),
                    "source": msg.get("source"),
                    "exchange": msg.get("exchange"),
                    "captured_at": msg.get("captured_at"),
                    "has_payload": "payload" in msg,
                    "payload": msg.get("payload"),
                }
            )

    print(f"Consumer group: {group_id}")
    print(f"Consumed messages: {len(rows)}")
    if rows:
        display(pd.DataFrame(rows).head(20))
    else:
        print("No messages consumed. Check broker/topic connectivity or DO_INGEST=True.")
else:
    rows = []


if DO_CONSUME_TO_CLICKHOUSE:
    inserted = consume_service.consume_polymarket_markets_raw_to_clickhouse(
        max_messages_per_partition=200,
        max_workers=3,
    )
    print("Inserted market snapshot rows to ClickHouse:", inserted)


2026-03-21 17:29:43 | INFO | app.services.ingestion_service | Fetching Polymarket markets for Kafka publish (topic=polymarket.markets.raw): {'sort': 'volume', 'limit': 200}
2026-03-21 17:29:43 | INFO | app.clients.polymarket_client | Fetching markets from Polymarket with params {'sort': 'volume', 'limit': 200}
2026-03-21 17:30:02 | INFO | app.services.ingestion_service | Publishing 200 messages to Kafka topic 'polymarket.markets.raw'
2026-03-21 17:30:02 | INFO | app.clients.kafka_client | Initializing KafkaProducerClient (bootstrap_servers=['localhost:9092'])
2026-03-21 17:30:02 | INFO | app.clients.kafka_client | Opening Kafka producer connection
2026-03-21 17:30:03 | INFO | app.clients.kafka_client | Sending batch of 200 messages to topic 'polymarket.markets.raw'
2026-03-21 17:30:03 | INFO | app.clients.kafka_client | Batch send completed (200 messages -> topic 'polymarket.markets.raw')
2026-03-21 17:30:03 | INFO | app.clients.kafka_client | Flushing Kafka producer
2026-03-21 17:30:0

,partition,market_id,source,exchange,captured_at,has_payload,payload
0,2,561973,pmxt,polymarket,2026-03-21T07:35:51.406617Z,True,"{'market_id': '561973', 'title': 'Republican P..."
1,2,561973,pmxt,polymarket,2026-03-21T07:36:44.250203Z,True,"{'market_id': '561973', 'title': 'Republican P..."
2,2,561973,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561973', 'title': 'Republican P..."
3,2,562006,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '562006', 'title': 'Republican P..."
4,2,561998,pmxt,polymarket,2026-03-21T07:39:04.229765Z,True,"{'market_id': '561998', 'title': 'Republican P..."
5,2,561973,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '561973', 'title': 'Republican P..."
6,2,562006,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '562006', 'title': 'Republican P..."
7,2,561998,pmxt,polymarket,2026-03-21T07:45:04.589695Z,True,"{'market_id': '561998', 'title': 'Republican P..."
8,2,559664,pmxt,polymarket,2026-03-21T07:47:14.008469Z,True,"{'market_id': '559664', 'title': 'Democratic P..."
9,2,561996,pmxt,polymarket,2026-03-21T07:47:14.008469Z,True,"{'market_id': '561996', 'title': 'Republican P..."


2026-03-21 17:30:38 | INFO | app.services.consume_service | Consuming Kafka topic 'polymarket.markets.raw' into ClickHouse (group_id=polymarket.markets.raw.to.clickhouse.v1)
2026-03-21 17:30:38 | INFO | app.clients.kafka_client | Initializing KafkaConsumerClient (topic=polymarket.markets.raw, group_id=polymarket.markets.raw.to.clickhouse.v1)
2026-03-21 17:30:38 | INFO | app.clients.kafka_client | Opening Kafka consumer connection (topic=polymarket.markets.raw, group_id=polymarket.markets.raw.to.clickhouse.v1)
2026-03-21 17:30:39 | INFO | app.clients.kafka_client | Starting scheduled bounded consumption (topic=polymarket.markets.raw, partitions=[0, 1, 2], max_per_partition=200)
2026-03-21 17:30:39 | INFO | app.clients.kafka_client | Partition 1 bounded consume (start=3250, stop=3450, end=21653)
2026-03-21 17:30:39 | INFO | app.clients.kafka_client | Partition 2 bounded consume (start=3284, stop=3484, end=22051)
2026-03-21 17:30:39 | INFO | app.clients.kafka_client | Partition 0 bounded 

In [5]:
# Kafka 单条 payload 示例（便于核对字段）
if rows:
    pd.DataFrame(rows)["payload"].iloc[0]
else:
    print("No rows in current notebook run.")

In [13]:
# 查询新 schema 的入库结果
for table in [
    "polymarket_market_dim",
    "polymarket_market_snapshot",
    "polymarket_outcome_snapshot",
]:
    count_df = ch_client.query_df(f"SELECT count() AS cnt FROM {table}")
    print(f"\n{table} total rows:")
    display(count_df)

# print("\nLatest market dim (captured_at):")
# display(
#     ch_client.query_df(
#         """
#         SELECT *
#         FROM polymarket_market_dim
#         LIMIT 10
#         """
#     )
# )

print("\nLatest market snapshots (captured_at):")
display(
    ch_client.query_df(
        """
        SELECT market_id, captured_at, volume24h, volume, liquidity, open_interest
        FROM polymarket_market_snapshot
        ORDER BY captured_at DESC
        LIMIT 10
        """
    )
    .sort_values(by = 'market_id')
)

print("\nLatest outcome snapshots (captured_at):")
display(
    ch_client.query_df(
        """
        SELECT market_id, outcome_id, label, captured_at, price, price_change24h
        FROM polymarket_outcome_snapshot
        ORDER BY captured_at DESC
        LIMIT 10
        """
    )
    .sort_values(by = 'market_id')
)



polymarket_market_dim total rows:


,cnt
0,2016



polymarket_market_snapshot total rows:


,cnt
0,9000



polymarket_outcome_snapshot total rows:


,cnt
0,18000



Latest market snapshots (captured_at):


,market_id,captured_at,volume24h,volume,liquidity,open_interest
0,1598807,2026-03-21 08:10:03,20483.104716,29091.568422,22103.12106,0.0
1,1598813,2026-03-21 08:10:03,57590.437106,68168.297065,21533.25633,0.0
6,1598816,2026-03-21 08:10:03,62349.670954,74054.880814,58536.52502,0.0
3,1603540,2026-03-21 08:10:03,34651.023093,100015.063967,36309.84870,0.0
4,1603549,2026-03-21 08:10:03,22554.389789,160478.206897,28210.75422,0.0
5,1603559,2026-03-21 08:10:03,60345.822754,209031.090672,23521.08720,0.0
7,1603561,2026-03-21 08:10:03,51486.340960,197419.739174,42334.70053,0.0
9,1603566,2026-03-21 08:10:03,10490.070436,69329.206031,19572.24185,0.0
8,1605699,2026-03-21 08:10:03,20652.406921,23967.473076,237225.78415,0.0
2,1605701,2026-03-21 08:10:03,12437.879638,15397.923253,242053.80404,0.0



Latest outcome snapshots (captured_at):


,market_id,outcome_id,label,captured_at,price,price_change24h
2,1598804,5151380878790488454472371743449022444965036800...,"72,000",2026-03-21 08:10:03,0.1700,0.0
8,1598804,9472648120016879930216038050925044111562558497...,"Not 72,000",2026-03-21 08:10:03,0.8300,0.0
7,1598813,4418254293990128450103189273196547031317385556...,"80,000",2026-03-21 08:10:03,0.0025,0.0
9,1598813,2233255920907405613426991303183086654996676135...,"Not 80,000",2026-03-21 08:10:03,0.9975,0.0
4,1598816,5277008534534361509144714848120803044789653257...,"Not 82,000",2026-03-21 08:10:03,0.9990,0.0
5,1598816,3898449562714574504142259304002421245331032274...,"82,000",2026-03-21 08:10:03,0.0010,0.0
3,1603540,3356953676265541463956803178177592599816747917...,"Not ↑ 88,000",2026-03-21 08:10:03,0.9995,0.0
6,1603540,5162068244365828830961939440987203693216482779...,"↑ 88,000",2026-03-21 08:10:03,0.0005,0.0
0,1603549,9650835601538750287742871907043794478436378916...,"Not ↑ 80,000",2026-03-21 08:10:03,0.9940,0.0
1,1603549,1106125692060784840476231483522233177205571993...,"↑ 80,000",2026-03-21 08:10:03,0.0060,0.0
